In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget 
from scipy.optimize import root_scalar

# Define the equation f(u, a) = 0
def f(u, a):
    return (u**2 + a*u + 1)**2 - 1*(u**2 - 1)

# Compute s from u and a
def compute_s(u, a):
    return u / (u**2 + u*a + 1) - u

# Range of a values
a_values = np.linspace(-6, 0, 200)

# Storage for results
a_list, s_list = [], []

for a_val in a_values:
    # Scan u >= 1 using small intervals to detect sign changes
    u_grid = np.logspace(0, 2, 100)
    f_vals = f(u_grid, a_val)
    
    # Find sign changes between consecutive points
    for i in range(len(u_grid) - 1):
        if f_vals[i] * f_vals[i+1] < 0:
            try:
                sol = root_scalar(f, args=(a_val,), method='brentq',
                                  bracket=[u_grid[i], u_grid[i+1]])
                if sol.converged:
                    u_root = sol.root
                    s_val = compute_s(u_root, a_val)
                    a_list.append(a_val)
                    s_list.append(s_val)
            except ValueError:
                # No root in bracket
                pass

# Plot
fig = plt.figure(figsize=(3,4))
ax = fig.add_subplot()
ax.plot(a_list, s_list, '.', color='blue')
ax.set_xlabel(r"$z_a$")
ax.set_ylabel(r"$z_s$")
ax.set_xlim([-6, 0])
ax.set_ylim([-6, 6])
ax.plot(a_values, a_values, 'k--', linewidth=1)
ax.vlines(-2, -10, 10, colors='k', linestyles='--', linewidth=1)
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

# Parameters
b = 1
a_vals = np.linspace(-5, 5, 100)
s_vals = np.linspace(-5, 5, 100)

# Define the two equations
def f1(u, a, s, b):
    return (a + u + 1/u) * (s + u) - b

def f2(u, a, s, b):
    return (a - u - 1/u) * (s - u) - b

def count_roots(a, s, b):
    """Count distinct real roots of both equations for given a, s."""
    u_grid = np.logspace(0, 2, 100)
    roots = []

    for func in [f1, f2]:
        fvals = func(u_grid, a, s, b)
        for i in range(len(u_grid) - 1):
            if np.isnan(fvals[i]) or np.isnan(fvals[i+1]):
                continue
            if fvals[i] * fvals[i+1] < 0:
                try:
                    sol = root_scalar(func, args=(a, s, b),
                                      bracket=[u_grid[i], u_grid[i+1]],
                                      method='brentq')
                    if sol.converged:
                        u_root = sol.root
                        # avoid duplicates
                        if not any(abs(u_root - r) < 1e-4 for r in roots):
                            roots.append(u_root)
                except Exception:
                    pass
    return len(roots)

# Precompute solution counts
counts = np.zeros((len(a_vals), len(s_vals)), dtype=int)

for i, a_val in enumerate(a_vals):
    for j, s_val in enumerate(s_vals):
        counts[i, j] = count_roots(a_val, s_val, b)

# Plot results
plt.figure(figsize=(5,4))
plt.pcolormesh(a_vals, s_vals, counts.T, shading='auto', cmap='coolwarm')
plt.xlabel(r"$z_a$")
plt.ylabel(r"$z_s$")
plt.xlim([-5, 5])
plt.ylim([-5, 5])
plt.plot(a_list, s_list, '.', color='k')
plt.vlines([-2, 2], -10, 10, colors='k', linestyles='--', linewidth=1)
plt.hlines([-1, 1], xmin=-10, xmax=10, color='k', linestyles='--', linewidth=1)
cbar = plt.colorbar()
cbar.set_label('Number of solutions')
cbar.set_ticks([0, 1, 2])
plt.savefig('existence.png', dpi=200)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from scipy.optimize import root_scalar
import plotly.graph_objects as go

# Define parameter ranges
b_values = [0, 1, 2, 3, 4, 5]
a_vals = np.linspace(-5, 5, 50)
s_vals = np.linspace(-5, 5, 50)
u_min, u_max = 1.0, 20.0

# Equations
def f1(u, a, s, b):
    return (a + u + 1/u) * (s + u) - b

def f2(u, a, s, b):
    return (a - u - 1/u) * (s - u) - b

def count_roots_single(func, a, s, b):
    """Count distinct real roots of one equation for given a, s."""
    u_grid = np.linspace(u_min, u_max, 150)
    fvals = func(u_grid, a, s, b)
    roots = []
    for i in range(len(u_grid) - 1):
        if np.isnan(fvals[i]) or np.isnan(fvals[i+1]):
            continue
        if fvals[i] * fvals[i+1] < 0:
            try:
                sol = root_scalar(func, args=(a, s, b),
                                  bracket=[u_grid[i], u_grid[i+1]],
                                  method='brentq')
                if sol.converged:
                    u_root = sol.root
                    if not any(abs(u_root - r) < 1e-4 for r in roots):
                        roots.append(u_root)
            except Exception:
                pass
    return len(roots)

def color_for_counts(n1, n2):
    """Return RGB color based on number of f1 and f2 solutions."""
    blue_light = np.array([0.6, 0.8, 1.0])
    blue_dark  = np.array([0.0, 0.2, 0.8])
    red_light  = np.array([1.0, 0.7, 0.7])
    red_dark   = np.array([0.7, 0.0, 0.0])
    white      = np.array([1.0, 1.0, 1.0])

    if n1 == 1: c1 = blue_light
    elif n1 >= 2: c1 = blue_dark
    else: c1 = white

    if n2 == 1: c2 = red_light
    elif n2 >= 2: c2 = red_dark
    else: c2 = white

    if np.allclose(c1, white): return c2
    if np.allclose(c2, white): return c1
    return 0.5 * (c1 + c2)

# Precompute all frames
frames = []
for b in b_values:
    colors = np.zeros((len(a_vals), len(s_vals), 3))
    for i, a_val in enumerate(a_vals):
        for j, s_val in enumerate(s_vals):
            n1 = count_roots_single(f1, a_val, s_val, b)
            n2 = count_roots_single(f2, a_val, s_val, b)
            colors[i, j] = color_for_counts(n1, n2)
    frames.append(colors)

# Create Plotly frames for slider
plotly_frames = []
for k, b in enumerate(b_values):
    frame_img = (frames[k] * 255).astype(np.uint8)
    plotly_frames.append(go.Frame(
        data=[go.Image(z=frame_img)],
        name=f"b={b}"
    ))

# Initial figure
fig = go.Figure(
    data=[go.Image(z=(frames[0]*255).astype(np.uint8))],
    frames=plotly_frames
)

# Slider
sliders = [{
    "pad": {"t": 30},
    "x": 0.1, "len": 0.8,
    "active": 0,
    "steps": [
        {
            "args": [[f"b={b}"], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}],
            "label": f"b={b}",
            "method": "animate"
        }
        for b in b_values
    ]
}]

# Layout
fig.update_layout(
    title="Solution regions in (a,s) plane as function of eta",
    xaxis_title="z_s",
    yaxis_title="z_a",
    sliders=sliders,
)

# Aspect and axes
fig.update_yaxes(autorange="reversed")  # match image coordinates
fig.update_xaxes(autorange="reversed")  # match image coordinates
fig.update_layout(height=600, width=600)

fig.show()
